In [ ]:
#!pip install regex

# Lecture et Nettoyage du Texte

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import regex as rex
import time

with open('data/corpus_Moliere.txt','r',encoding='utf-8') as f:
    text_raw = f.read()
print('Longueur du texte en nombre de caractères :', len(text_raw))
print(text_raw[:1000])

Longueur du texte en nombre de caractères : 1655683
LE MÉDECIN VOLANT[3]

COMÉDIE


Des personnages dont le caractère est convenu, le costume arrêté
d'avance, le langage différent, le type invariable, et qui, sur un plan
tracé, improvisent un dialogue pittoresque, conforme aux situations,
telle est la comédie «all' improviso» que les Italiens ont inventée;
celle que Trivelin, Scaramouche et Mezzetin ont fait applaudir en
France. La souplesse physique et la facilité du dialogue prêtent, si ce
n'est de la valeur, au moins du charme à cette vive forme de l'art,
forme enfantine, la seule qui, au commencement du dix-septième siècle et
à la fin du seizième, fût populaire dans le midi de l'Europe.

  [3] Le titre de l'arlequinade italienne est: _Il Medico volante_, le
  _Médecin sauteur_; épithète justifiée par les singuliers tours de force
  que le héros de la farce accomplit.

Poquelin enfant, lorsqu'il allait du collége de Clermont aux
Saints-Innocents et de la halle au collége, dut admire

In [2]:
#nettoyage du texte
import re
def preprocess_text(_text):
    """Nettoie un fichier texte pour l'entraînement NLP"""
        
    # Stats avant nettoyage
    original_length = len(_text)
    original_lines = _text.count('\n')
    
    # Nettoyage
    _text = re.sub(r'[ \t]+', ' ', _text)              # Espaces multiples
    _text = re.sub(r'\n{3,}', '\n\n', _text)           # Lignes vides
    _text = re.sub(r'(?m)^[ \t]+|[ \t]+$', '', _text)  # Espaces début/fin ligne
    _text = re.sub(r'\s+([.,;:!?])', r'\1', _text)     # Espace avant ponctuation
    _text = _text.strip()

    # 1. Remplacements intelligents
    _text = _text.replace('—', '-')
    _text = _text.replace('«', '"').replace('»', '"')
    _text = _text.replace('[', '(').replace(']', ')')
    _text = _text.replace('{', '(').replace('}', ')')
    _text = _text.replace(''', "'").replace(''', "'")
    _text = _text.replace('…', '...')
    _text = _text.replace('«', '"').replace('»', '"')
    
    # 2. Filtrer caractères
    allowed = set(
        'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
        '0123456789 .,;:!?\'\"-\n'
        'àâçèéêëîïôùûÀÂÇÉÈÊËÎÏÔÙÛœŒ'
        '()'
    )
    _text = ''.join(c for c in _text if c in allowed)
    
    # 3. Normaliser espaces
    _text = re.sub(r' +', ' ', _text)
    _text = re.sub(r'\n{3,}', '\n\n', _text)
    _text = re.sub(r'(?m)^[ ]+|[ ]+$', '', _text)
    
    print(f"Texte nettoyé: {len(_text):,} caractères")
    
    _text = _text.strip()    
    # Stats après nettoyage
    cleaned_length = len(_text)
    cleaned_lines = _text.count('\n')
    
    print(f"Nettoyage terminé:")
    print(f"  Caractères: {original_length:,} → {cleaned_length:,} ({cleaned_length/original_length*100:.1f}%)")
    print(f"  Lignes: {original_lines:,} → {cleaned_lines:,}")
    
    return _text

In [3]:
texte_clean = preprocess_text(text_raw)

Texte nettoyé: 1,571,455 caractères
Nettoyage terminé:
  Caractères: 1,655,683 → 1,571,455 (94.9%)
  Lignes: 59,017 → 57,924


# Encodage

In [5]:
# get all unique characters in the text
def getDictUnicode(_text):
    _chars = sorted(list(set(_text)))
    _vocab_size = len(_chars)
    print('Nombre de caractères uniques :', _vocab_size)
    print('Liste des caractères uniques :', ''.join(_chars))
    return _chars, _vocab_size


In [6]:
def getStats(ids,c={}):
    count = c.copy()
    for pair in zip(ids, ids[1:]):
        count[pair] = count.get(pair, 0) + 1
    return count

def merge(ids, pair, idx):
    i = 0
    merged = []
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged.append(idx)
            i += 2
        else:
            merged.append(ids[i])
            i += 1
    return merged

In [7]:
def decode(ids,vocab):
    decoded_bytes = bytearray()
    for idx in ids:
        decoded_bytes.extend(vocab[idx])
    return decoded_bytes.decode('utf-8', errors='replace')

def encode(text,vocab):
    tokens = text.encode("utf-8") #raw byte
    ids = list(map(int,tokens)) #convertion liste d'entiers INT
    # Apply BPE merges
    i = 0
    while i < len(ids):
        merged = False
        for (p0, p1), idx in merges.items():
            if i < len(ids) - 1 and ids[i] == p0 and ids[i+1] == p1:
                ids[i] = idx
                del ids[i+1]
                merged = True
                break
        if not merged:
            i += 1
    return ids

def encode2(text,vocab):
    tokens = list(text.encode("utf-8")) #raw byte
    while len(tokens) > 1:
        stats = getStats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float('inf')))
        if not pair in merges:
            break # No more merges available
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

def decode2(ids,vocab):
    tokens = b''.join(vocab[idx] for idx in ids)
    return tokens.decode('utf-8', errors='replace')

In [8]:
text = texte_clean
chars, vocab_size = getDictUnicode(text)

Nombre de caractères uniques : 94
Liste des caractères uniques : 
 !"'(),-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVXYZabcdefghijklmnopqrstuvwxyzÇÈÉÊÏÛàâçèéêëîïôùûŒœ


In [9]:
tokens = texte_clean.encode("utf-8") # raw byte
tokens = list(map(int,tokens)) # convertion liste d'entiers INT
print('Nombre de tokens :', len(tokens))
stats = getStats(tokens)
print('Statistiques des bigrams :', stats)
print('Nombre de bigrams uniques :', len(stats))

Nombre de tokens : 1607524
Statistiques des bigrams : {(76, 69): 2461, (69, 32): 1679, (32, 77): 1483, (77, 195): 432, (195, 137): 2211, (137, 68): 40, (68, 69): 627, (69, 67): 21, (67, 73): 342, (73, 78): 950, (78, 32): 708, (32, 86): 699, (86, 79): 46, (79, 76): 583, (76, 65): 539, (65, 78): 2458, (78, 84): 756, (84, 40): 1, (40, 51): 201, (51, 41): 159, (41, 10): 103, (10, 10): 19053, (10, 67): 2009, (67, 79): 133, (79, 77): 108, (68, 73): 147, (73, 69): 905, (69, 10): 71, (10, 68): 2247, (68, 101): 666, (101, 115): 19057, (115, 32): 34499, (32, 112): 18403, (112, 101): 3710, (101, 114): 10957, (114, 115): 2878, (115, 111): 5521, (111, 110): 17310, (110, 110): 2710, (110, 97): 1306, (97, 103): 1795, (103, 101): 2713, (32, 100): 24238, (100, 111): 3147, (110, 116): 14053, (116, 32): 26835, (32, 108): 17289, (108, 101): 16122, (101, 32): 57963, (32, 99): 14872, (99, 97): 1312, (97, 114): 6929, (114, 97): 6069, (97, 99): 1722, (99, 116): 720, (116, 195): 2635, (195, 168): 3510, (168, 1

In [10]:
tokens = text.encode("utf-8") #raw byte
ids = list(tokens)
print(ids[:10])
addedToken = 40  # Nombre de tokens désirés (bytes 0-255)
merges = {}
for i in range(addedToken):
    s = getStats(ids)
    pair = max(s, key=s.get)
    if not s:
        break
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx
#    print(f'Merged pair {pair} into token ID {idx}')
print('Nombre de tokens avant BPE :', len(tokens))
print('Nombre de tokens après BPE :', len(ids))
print(f'compression : {len(tokens)/len(ids):.2f}x')
print('Nombre de tokens uniques après BPE :', len(set(ids)))

[76, 69, 32, 77, 195, 137, 68, 69, 67, 73]
Nombre de tokens avant BPE : 1607524
Nombre de tokens après BPE : 1133030
compression : 1.42x
Nombre de tokens uniques après BPE : 134


In [11]:
print(merges)
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0,p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
print('Exemples de tokens dans le vocabulaire étendu :', vocab)

{(101, 32): 256, (115, 32): 257, (116, 32): 258, (111, 117): 259, (44, 32): 260, (101, 110): 261, (10, 10): 262, (111, 110): 263, (46, 262): 264, (113, 117): 265, (195, 169): 266, (97, 105): 267, (114, 32): 268, (111, 105): 269, (114, 101): 270, (97, 110): 271, (100, 256): 272, (101, 117): 273, (101, 257): 274, (101, 115): 275, (97, 32): 276, (259, 257): 277, (114, 256): 278, (108, 256): 279, (101, 114): 280, (97, 117): 281, (112, 97): 282, (69, 264): 283, (105, 32): 284, (265, 256): 285, (263, 32): 286, (117, 110): 287, (195, 160): 288, (44, 10): 289, (105, 108): 290, (101, 258): 291, (114, 105): 292, (288, 32): 293, (261, 32): 294, (118, 277): 295}
Exemples de tokens dans le vocabulaire étendu : {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x1

In [12]:
print(len(vocab), vocab)
test = (decode2(encode2(text,vocab),vocab))
print('Décodage correct :', test == text)

296 {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b

# Version Tokenization GPT

In [13]:
pattern  = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
pattern2 = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
pattern3 = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s"""
GPT4_SP  = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

def tokenize(_text,option=1):
    if (option==1):
        patG = rex.compile(pattern)
    elif (option==2):
        patG = rex.compile(pattern2)
    elif (option==3):
        patG = rex.compile(pattern3)
    else:
        patG = rex.compile(GPT4_SP)

    tokens = rex.findall(patG, _text)
    print(f'Nombre de tokens : {len(tokens):,}')
    return tokens

text_tk = texte_clean
start = time.time()
t_ = tokenize(text_tk,3)
end = time.time()
print(f'Temps de tokenization : {end - start:.2f} secondes')
tk_ = [ list(map(int,i.encode('utf-8'))) for i in t_ ]
print(tk_[:10])

Nombre de tokens : 357,887
Temps de tokenization : 0.15 secondes
[[76, 69], [32, 77, 195, 137, 68, 69, 67, 73, 78], [32, 86, 79, 76, 65, 78, 84], [40], [51], [41, 10, 10], [67, 79, 77, 195, 137, 68, 73, 69], [10, 10], [68, 101, 115], [32, 112, 101, 114, 115, 111, 110, 110, 97, 103, 101, 115]]


In [14]:
def getStats_tk(ids,c={}):
    count = c.copy()
    for pair in zip(ids, ids[1:]):
        count[pair] = count.get(pair, 0) + 1
    return count

def maxStats(ids,c={}):
    count = c.copy()
    for k in ids:
        count = getStats_tk(k,count)
    return max(count, key=count.get)

def merge_tk(ids_, pair, idx):
    merged = []
    print(pair, idx)
    for row in ids_:
        merged_row = []
        i = 0
        while i < len(row):
            if i < len(row) - 1 and (row[i], row[i+1]) == pair:
                merged_row.append(idx)
                i += 2
            else:
                merged_row.append(row[i])
                i += 1
        merged.append(merged_row)
    return merged

In [15]:
s_ = {}
for i in tk_:
    s_ = getStats_tk(i,s_)
print(max(s_, key=s_.get))
print(sorted(s_.items(), key=lambda x: x[1], reverse=True)[:5])

(32, 100)
[((32, 100), 24238), ((111, 117), 23203), ((101, 110), 19255), ((114, 101), 19083), ((101, 115), 19057)]


In [ ]:
ids_tk = list(tk_)
print(ids_tk[:10])
addedToken = 3  # Nombre de tokens désirés (bytes 0-255)
tk_merges = {}
for i in range(addedToken):
    pair = maxStats(ids_tk, {})
    if not pair:
        break
    idx = 256 + i
    ids_tk = merge_tk(ids_tk, pair, idx)
    tk_merges[pair] = idx
#    print(f'Merged pair {pair} into token ID {idx}')
lenInit = sum(len(row) for row in tk_)
lenIDS = sum(len(row) for row in ids_tk)
lenInit = sum(len(row) for row in tk_)
valUnique = set(val for row in ids_tk for val in row)
print('Nombre de tokens avant BPE :', lenInit)
print('Nombre de tokens après BPE :', lenIDS)
print(f'compression : {lenInit/lenIDS:.2f}x')
print('Nombre de tokens uniques après BPE :', len(valUnique))

In [ ]:
print(tk_[:10])

In [131]:
from collections import Counter, defaultdict
from itertools import chain

def getStats_tk_optimized(ids):
    """Compte les paires sans copier le dictionnaire"""
    pairs = Counter()
    for row in ids:
        for pair in zip(row, row[1:]):
            pairs[pair] += 1
    return pairs

def maxStats_optimized(ids):
    """Récupère directement la paire la plus fréquente"""
    pairs = getStats_tk_optimized(ids)
    return pairs.most_common(1)[0][0] if pairs else None

def merge_tk_optimized(ids_, pair, idx):
    """Version optimisée du merge avec des listes pré-allouées"""
    merged = []
    pair_tuple = pair  # Évite les recherches répétées
    
    for row in ids_:
        merged_row = []
        i = 0
        row_len = len(row)
        
        while i < row_len:
            # Vérification rapide sans créer un tuple à chaque fois
            if i < row_len - 1 and row[i] == pair_tuple[0] and row[i+1] == pair_tuple[1]:
                merged_row.append(idx)
                i += 2
            else:
                merged_row.append(row[i])
                i += 1
        merged.append(merged_row)
    
    return merged

# === Code principal optimisé ===

ids_tk = list(tk_)
print(ids_tk[:10])

addedToken = 100  # Nombre de tokens désirés (bytes 0-255)
tk_merges = {}

# Calcul de la taille initiale une seule fois
lenInit = sum(len(row) for row in ids_tk)

for i in range(addedToken):
    pair = maxStats_optimized(ids_tk)
    
    if not pair:
        break
    
    idx = 256 + i
    ids_tk = merge_tk_optimized(ids_tk, pair, idx)
    tk_merges[pair] = idx

# Calculs finaux
lenIDS = sum(len(row) for row in ids_tk)
valUnique = set(chain(*ids_tk))  # Utiliser chain au lieu de boucles imbriquées

print('Nombre de tokens avant BPE :', lenInit)
print('Nombre de tokens après BPE :', lenIDS)
print(f'Compression : {lenInit/lenIDS:.2f}x')
print('Nombre de tokens uniques après BPE :', len(valUnique))

[[76, 69], [32, 77, 195, 137, 68, 69, 67, 73, 78], [32, 86, 79, 76, 65, 78, 84], [40], [51], [41, 10, 10], [67, 79, 77, 195, 137, 68, 73, 69], [10, 10], [68, 101, 115], [32, 112, 101, 114, 115, 111, 110, 110, 97, 103, 101, 115]]
Nombre de tokens avant BPE : 1607524
Nombre de tokens après BPE : 959410
Compression : 1.68x
Nombre de tokens uniques après BPE : 191


In [128]:
print(ids_tk[:10])

[[335], [399, 348, 644, 67, 614], [501, 712, 2414], [40], [51], [41, 260], [3779], [260], [1750], [4166]]


In [115]:
def decodeTK(tokens_, vocab_):
    result = ""
    # Traiter chaque ligne du tableau de tokens
    for row in tokens_:
        byte_tokens = list(row)
        result += decode(byte_tokens,vocab_)
    return result

In [116]:
import json, base64

def save_vocab(vocab_, filepath='data/vocab.json'):
    vocab_dict = {}    
    for idx, byte_val in vocab_.items():
        vocab_dict[str(idx)] = base64.b64encode(byte_val).decode('ascii')
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(vocab_dict, f, indent=2, ensure_ascii=False)
    print(f"Vocabulaire sauvegardé dans {filepath}")

def load_vocab(filepath='data/vocab.json'):
    with open(filepath, 'r', encoding='utf-8') as f:
        vocab_dict = json.load(f)
    vocab = {}
    for idx_str, string_val in vocab_dict.items():
        vocab[int(idx_str)] = base64.b64decode(string_val)
    print(f"Vocabulaire chargé depuis {filepath}")
    return vocab
    

vocabTK = {idx: bytes([idx]) for idx in range(256)}
for (p0,p1), idx in tk_merges.items():
    vocabTK[idx] = vocabTK[p0] + vocabTK[p1]
print('Exemples de tokens dans le vocabulaire étendu :', vocabTK)
save_vocab(vocabTK)


Exemples de tokens dans le vocabulaire étendu : {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 8

In [117]:
vocabLoaded = load_vocab()
print('Exemples de tokens dans le vocabulaire chargé :', vocabLoaded)
print(vocabTK)
print(vocabLoaded == vocabTK)
for k in vocabTK:
    if vocabTK[k] != vocabLoaded[k]:
        print(f'Différence pour le token {k}: {vocabTK[k]} vs {vocabLoaded[k]}')

Vocabulaire chargé depuis data/vocab.json
Exemples de tokens dans le vocabulaire chargé : {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R',

In [118]:
ids_arr = list(chain(*ids_tk))
print("\nSimple array:", ids_arr[:100])
print("en clair:", decode(ids_arr, vocabLoaded))


Simple array: [335, 399, 348, 644, 67, 614, 501, 712, 2414, 40, 51, 41, 260, 3779, 260, 1750, 4166, 654, 307, 3780, 369, 2051, 1803, 44, 307, 268, 375, 2002, 309, 2164, 269, 10, 100, 499, 584, 44, 307, 2385, 3572, 288, 44, 307, 281, 1804, 101, 4167, 287, 105, 497, 44, 296, 345, 44, 476, 317, 364, 277, 10, 454, 4006, 44, 1849, 111, 118, 297, 288, 317, 740, 355, 4168, 261, 314, 2441, 259, 382, 44, 996, 301, 309, 683, 266, 2607, 1960, 294, 293, 465, 369, 304, 1164, 1243, 1544, 39, 1849, 111, 118, 297, 111, 34, 290, 340]
en clair: LE MÉDECIN VOLANT(3)

COMÉDIE

Des personnages dont le caractère est convenu, le costume arrêté
d'avance, le langage différent, le type invariable, et qui, sur un plan
tracé, improvisent un dialogue pittoresque, conforme aux situations,
telle est la comédie "all' improviso" que les Italiens ont inventée;
celle que Trivelin, Scaramouche et Mezzetin ont fait applaudir en
France. La souplesse physique et la facilité du dialogue prêtent, si ce
n'est de la valeur, au

In [119]:
def encodeTK(input_, vocab_):
    # Créer un dictionnaire inverse {bytes_sequence: token_id}
    inverse_vocab = {v: k for k, v in vocab_.items()}
    
    # Trier par longueur décroissante pour matcher les séquences les plus longues d'abord
    sorted_pairs = sorted(inverse_vocab.items(), key=lambda x: len(x[0]), reverse=True)
    
    encoded_array = []
    
    # Traiter chaque ligne du tableau
    for row in input_:
        # Convertir la liste d'entiers en bytes
        row_bytes = bytes(row)
        encoded_row = []
        i = 0
        # Parcourir les bytes et chercher les tokens correspondants
        while i < len(row_bytes):
            matched = False
            # Essayer les tokens les plus longs d'abord
            for token_bytes, token_id in sorted_pairs:
                # Vérifier si les bytes actuels correspondent au token
                if row_bytes[i:i+len(token_bytes)] == token_bytes:
                    encoded_row.append(token_id)
                    i += len(token_bytes)
                    matched = True
                    break
            
            # Si pas de correspondance, prendre 1 byte à la fois
            if not matched:
                encoded_row.append(row_bytes[i:i+1][0])
                i += 1
        
        encoded_array.append(encoded_row)
    
    return encoded_array

In [130]:
print(tk_[:100])

[[76, 69], [32, 77, 195, 137, 68, 69, 67, 73, 78], [32, 86, 79, 76, 65, 78, 84], [40], [51], [41, 10, 10], [67, 79, 77, 195, 137, 68, 73, 69], [10, 10], [68, 101, 115], [32, 112, 101, 114, 115, 111, 110, 110, 97, 103, 101, 115], [32, 100, 111, 110, 116], [32, 108, 101], [32, 99, 97, 114, 97, 99, 116, 195, 168, 114, 101], [32, 101, 115, 116], [32, 99, 111, 110, 118, 101, 110, 117], [44], [32, 108, 101], [32, 99, 111, 115, 116, 117, 109, 101], [32, 97, 114, 114, 195, 170, 116, 195, 169], [10], [100], [39, 97, 118, 97, 110, 99, 101], [44], [32, 108, 101], [32, 108, 97, 110, 103, 97, 103, 101], [32, 100, 105, 102, 102, 195, 169, 114, 101, 110, 116], [44], [32, 108, 101], [32, 116, 121, 112, 101], [32, 105, 110, 118, 97, 114, 105, 97, 98, 108, 101], [44], [32, 101, 116], [32, 113, 117, 105], [44], [32, 115, 117, 114], [32, 117, 110], [32, 112, 108, 97, 110], [10], [116, 114, 97, 99, 195, 169], [44], [32, 105, 109, 112, 114, 111, 118, 105, 115, 101, 110, 116], [32, 117, 110], [32, 100, 105, 

In [120]:
text_encode =encodeTK(tk_,vocabLoaded)
text_encode_array = list(chain(*text_encode))
print(text_encode[:10])
print(text_encode == ids_tk)

[[335], [399, 2849, 69, 67, 614], [2897, 739, 3363], [40], [51], [1845, 10], [3779], [260], [1750], [4166]]
False


In [121]:
text_encode_array[:10]
print(decode(text_encode_array, vocabLoaded)[:100])

LE MÉDECIN VOLANT(3)

COMÉDIE

Des personnages dont le caractère est convenu, le costume arrêté
d'av


In [122]:
print(ids_tk[:10])
print(text_encode[:10])

[[335], [399, 348, 644, 67, 614], [501, 712, 2414], [40], [51], [41, 260], [3779], [260], [1750], [4166]]
[[335], [399, 2849, 69, 67, 614], [2897, 739, 3363], [40], [51], [1845, 10], [3779], [260], [1750], [4166]]


In [123]:
l = 9
print(ids_tk[:l])
print(text_encode[:l])
print(decodeTK(ids_tk[:l], vocabLoaded))
print(decodeTK(text_encode[:l], vocabLoaded))
print(vocabLoaded[259])

[[335], [399, 348, 644, 67, 614], [501, 712, 2414], [40], [51], [41, 260], [3779], [260], [1750]]
[[335], [399, 2849, 69, 67, 614], [2897, 739, 3363], [40], [51], [1845, 10], [3779], [260], [1750]]
LE MÉDECIN VOLANT(3)

COMÉDIE

Des
LE MÉDECIN VOLANT(3)

COMÉDIE

Des
b'es'


In [124]:
d1 = decodeTK(ids_tk, vocabLoaded)
d2 = decodeTK(text_encode, vocabLoaded)
print(d1 == d2)

True


In [125]:
t = [ decode(r, vocab=vocabLoaded) for r in text_encode[1000:2000] ]
txt = ''.join(t)
print(txt)

, médecin, monsieur! Je suis prêt à faire tout ce qu'il vous plaira;
mais, pour faire le médecin, je suis assez votre serviteur pour n'en
rien faire du tout; et par quel bout m'y prendre, bon Dieu? Ma foi,
monsieur, vous vous moquez de moi.

VALÈRE.

Si tu veux entreprendre cela, va, je te donnerai dix pistoles.

SGANARELLE.

Ah! pour dix pistoles, je ne dis pas que je ne sois médecin; car,
voyez-vous bien, monsieur, je n'ai pas l'esprit tant, tant subtil, pour
vous dire la vérité. Mais, quand je serai médecin, où irai-je?

VALÈRE.

Chez le bonhomme Gorgibus, voir sa fille qui est malade; mais tu es un
lourdaud qui, au lieu de bien faire, pourrois bien...

SGANARELLE.

Eh! mon Dieu, monsieur, ne soyez point en peine; je vous réponds que je
ferai aussi bien mourir une personne qu'aucun médecin qui soit dans la
ville. On dit un proverbe, d'ordinaire: Après la mort, le médecin; mais
vous verrez que, si je m'en mêle, on dira: Après le médecin, gare la
mort! Mais, néanmoins, quand je songe,

In [127]:
from IPython.display import HTML, display
import colorsys
import random
html = '<div style="font-family: monospace; font-size: 14px; line-height: 1.8;">'

t = [ decode(r, vocab=vocabLoaded) for r in text_encode[1000:1400] ]
txt = ''.join(t)

for text in t:
    # Générer une couleur aléatoire
    color = "#{:06x}".format(random.randint(0, 0xFFFFFF))
    html += f'<span style="color: #fff; background-color: {color}; padding: 2px 4px; margin: 2px; border-radius: 3px;">{text}</span>'
    
html += '</div>'
display(HTML(html))
print(f"\n✓ {len(t)} tokens affichés avec couleurs aléatoires")



✓ 400 tokens affichés avec couleurs aléatoires
